# Phase 03E: Teacher A Three-Seed Aggregate & Teacher B Entry

Notebook này đóng gói chính thức Teacher A sau ba seed và chỉ sau khi các
gate đạt mới phát hành quyền tạo Phase 03F Teacher B BLIP Preflight.

## Inputs bắt buộc

1. Phase 03A Teacher Pool Entry Gate.
2. Phase 03B Teacher A CLIP Preflight.
3. Teacher A seed 42 full-run output.
4. Teacher A seed 123 full-run output.
5. Teacher A seed 2026 full-run output.

## Nguyên tắc

- chỉ đọc Validation và metadata phát triển;
- không mở Held-out Test;
- không chọn lại checkpoint;
- best checkpoint của mỗi seed phải đúng marker đã khóa;
- Teacher B chưa được tải hoặc huấn luyện trong notebook này.


## Cell 1 — Environment và input discovery

### Công việc

- tạo output tree;
- tìm đúng ba COMPLETE marker của Teacher A;
- tìm Phase 03A entry contract và Phase 03B source contract;
- xác nhận seed `{42,123,2026}`;
- chưa đọc checkpoint hoặc dữ liệu.

### Thời gian dự kiến

5–30 giây.

### Tiến độ hiện tại

Cell 1/7 — bắt đầu kiểm kê và khám phá đầu vào.

### Đầu ra dự kiến

```text
Status                    : PASSED
Teacher A COMPLETE runs   : 3 / 3
Seeds                     : 42 / 123 / 2026
Test Set status           : SEALED
Progress                  : 1 / 7 cells passed
```


In [1]:
import csv
import hashlib
import json
import math
import os
import platform
import statistics
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path


CELL_STARTED_AT = time.perf_counter()
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = (
    Path("/kaggle/working")
    / ""
    / "phase3_teacher_a_three_seed_aggregate"
)
AUDIT_DIR = OUTPUT_ROOT / "audits"
CONFIG_DIR = OUTPUT_ROOT / "configs"
RESULT_DIR = OUTPUT_ROOT / "results"
MODEL_CARD_DIR = OUTPUT_ROOT / "model_cards"
MANIFEST_DIR = OUTPUT_ROOT / "manifests"
for directory in [
    OUTPUT_ROOT,
    AUDIT_DIR,
    CONFIG_DIR,
    RESULT_DIR,
    MODEL_CARD_DIR,
    MANIFEST_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def read_json(path):
    with open(path, "r", encoding="utf-8") as source:
        return json.load(source)


def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(
            payload,
            ensure_ascii=False,
            indent=2,
            sort_keys=True,
        )
        + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)


def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as source:
        for chunk in iter(lambda: source.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    candidate = (root / relative_path).resolve()
    assert candidate.is_relative_to(root)
    return candidate


def exactly_one(name):
    matches = sorted(INPUT_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name}, tìm thấy {len(matches)}: "
        f"{[str(path) for path in matches]}"
    )
    return matches[0].resolve()


assert INPUT_ROOT.is_dir()
teacher_markers = sorted(
    INPUT_ROOT.rglob("TEACHER_A_SEED*_FULL_RUN_COMPLETE.json")
)
assert len(teacher_markers) == 3, (
    "Cần gắn đúng ba output Teacher A seed 42/123/2026. "
    f"Tìm thấy: {[str(path) for path in teacher_markers]}"
)
marker_headers = [read_json(path) for path in teacher_markers]
assert {int(item["seed"]) for item in marker_headers} == {
    42,
    123,
    2026,
}

PHASE3A_READY_PATH = exactly_one(
    "PHASE3_TEACHER_POOL_INPUT_READY.json"
)
PHASE3A_ROOT = PHASE3A_READY_PATH.parent
PHASE3A_READY = read_json(PHASE3A_READY_PATH)
assert PHASE3A_READY["status"] == "ready"
assert PHASE3A_READY["test_set_status"] == "sealed"
assert PHASE3A_READY["test_set_used"] is False

PHASE3B_READY_PATH = exactly_one(
    "PHASE3_TEACHER_A_PREFLIGHT_READY.json"
)
PHASE3B_ROOT = PHASE3B_READY_PATH.parent
TEACHER_A_SOURCE_CONTRACT_PATH = exactly_one(
    "teacher_a_clip_pretrained_source_contract.json"
)
TEACHER_A_SOURCE_CONTRACT = read_json(
    TEACHER_A_SOURCE_CONTRACT_PATH
)
assert TEACHER_A_SOURCE_CONTRACT["status"] == "locked"
assert TEACHER_A_SOURCE_CONTRACT["test_set_status"] == "sealed"
assert TEACHER_A_SOURCE_CONTRACT["test_set_used"] is False

audit_path = AUDIT_DIR / "phase3e_cell01_input_discovery.json"
write_json(
    audit_path,
    {
        "schema_version": "phase3e_cell01_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_a_complete_markers": [
            str(path) for path in teacher_markers
        ],
        "seeds": [42, 123, 2026],
        "phase3a_ready": str(PHASE3A_READY_PATH),
        "phase3b_ready": str(PHASE3B_READY_PATH),
        "teacher_a_source_contract": str(
            TEACHER_A_SOURCE_CONTRACT_PATH
        ),
        "checkpoint_read": False,
        "validation_read": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03E CELL 1 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Python                    :", platform.python_version())
print("Teacher A COMPLETE runs   : 3 / 3")
print("Seeds                     : 42 / 123 / 2026")
print("Phase 03A                 : VERIFIED")
print("Phase 03B source contract : VERIFIED")
print("Checkpoint read           : False")
print("Validation read           : False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 7 cells passed")
print("Next                      : verify manifests and checkpoints")

PHASE 03E CELL 1 STATUS
Status                    : PASSED
Python                    : 3.12.13
Teacher A COMPLETE runs   : 3 / 3
Seeds                     : 42 / 123 / 2026
Phase 03A                 : VERIFIED
Phase 03B source contract : VERIFIED
Checkpoint read           : False
Validation read           : False
Test Set status           : SEALED
Progress                  : 1 / 7 cells passed
Next                      : verify manifests and checkpoints


## Cell 2 — COMPLETE marker, manifest và checkpoint integrity

### Công việc

- xác minh trạng thái/seed/epoch/step/quality gate;
- kiểm tra SHA-256 của best/last checkpoint;
- kiểm tra toàn bộ manifest và SHA256SUMS;
- kiểm tra container ZIP của checkpoint mà không load tensor.

### Thời gian dự kiến

2–8 phút, phụ thuộc tốc độ đọc khoảng 1,5 GB.

### Tiến độ hiện tại

Cell 2/7 — xác minh tính toàn vẹn của ba run Teacher A.

### Đầu ra dự kiến

```text
Status                    : PASSED
Manifest rows             : 95 / 95
Checkpoints               : 6 / 6 VERIFIED
Quality gates             : 3 / 3 PASSED
Progress                  : 2 / 7 cells passed
```


In [2]:
CELL_STARTED_AT = time.perf_counter()
EXPECTED_SEEDS = {42, 123, 2026}
run_records = []
manifest_rows_total = 0
sha_rows_total = 0
checkpoint_count = 0

for marker_path in teacher_markers:
    marker = read_json(marker_path)
    seed = int(marker["seed"])
    run_root = marker_path.parent.resolve()
    assert marker["status"] == "complete"
    assert marker["teacher_id"] == "A"
    assert marker["completed_epochs"] == 20
    assert marker["global_optimizer_steps"] == 35420
    assert marker["quality_gate_passed"] is True
    assert marker["qualified_for_kd"] is True
    assert marker["selection_split"] == "validation"
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

    checkpoint_records = {}
    for key in ["best_checkpoint", "last_checkpoint"]:
        record = marker[key]
        checkpoint = resolve_inside(
            run_root, record["relative_path"]
        )
        assert checkpoint.is_file()
        assert sha256_file(checkpoint) == record["sha256"]
        with zipfile.ZipFile(checkpoint) as archive:
            assert archive.testzip() is None
            assert archive.namelist()
        checkpoint_records[key] = {
            "path": str(checkpoint),
            "relative_path": record["relative_path"],
            "sha256": record["sha256"],
            "size_bytes": checkpoint.stat().st_size,
        }
        checkpoint_count += 1

    for manifest_key in ["csv", "json", "sha256sums"]:
        record = marker["manifests"][manifest_key]
        artifact = resolve_inside(
            run_root, record["relative_path"]
        )
        assert artifact.is_file()
        assert sha256_file(artifact) == record["sha256"]

    manifest_csv = resolve_inside(
        run_root, marker["manifests"]["csv"]["relative_path"]
    )
    with open(
        manifest_csv, "r", encoding="utf-8", newline=""
    ) as source:
        manifest_rows = list(csv.DictReader(source))
    for row in manifest_rows:
        artifact = resolve_inside(
            run_root, row["relative_path"]
        )
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256_file(artifact) == row["sha256"]
    manifest_rows_total += len(manifest_rows)

    sums_path = resolve_inside(
        run_root,
        marker["manifests"]["sha256sums"]["relative_path"],
    )
    sums_count = 0
    for line in sums_path.read_text(
        encoding="utf-8"
    ).splitlines():
        if not line.strip():
            continue
        sums_count += 1
        expected, relative_path = line.split("  ", 1)
        artifact = resolve_inside(run_root, relative_path)
        assert artifact.is_file()
        assert sha256_file(artifact) == expected
    assert sums_count == len(manifest_rows)
    sha_rows_total += sums_count

    best = marker["best_result"]
    assert best["prediction_count"] == 5000
    assert best["test_set_status"] == "sealed"
    assert best["test_set_used"] is False
    assert math.isfinite(float(best["teacher_forced_ce"]))
    assert all(
        math.isfinite(float(value))
        for value in best["metrics"].values()
    )
    run_records.append(
        {
            "teacher_id": "A",
            "seed": seed,
            "run_root": str(run_root),
            "completed_epochs": marker["completed_epochs"],
            "optimizer_steps": marker[
                "global_optimizer_steps"
            ],
            "best_epoch": int(best["epoch"]),
            "validation_ce": float(
                best["teacher_forced_ce"]
            ),
            "bleu_1": float(best["metrics"]["BLEU-1"]),
            "bleu_4": float(best["metrics"]["BLEU-4"]),
            "meteor": float(best["metrics"]["METEOR"]),
            "rouge_l": float(best["metrics"]["ROUGE-L"]),
            "cider_d": float(best["metrics"]["CIDEr-D"]),
            "quality_threshold": float(
                marker["quality_threshold"]
            ),
            "quality_margin": float(
                best["metrics"]["CIDEr-D"]
                - marker["quality_threshold"]
            ),
            "best_checkpoint": checkpoint_records[
                "best_checkpoint"
            ],
            "last_checkpoint": checkpoint_records[
                "last_checkpoint"
            ],
            "test_set_status": "sealed",
            "test_set_used": False,
        }
    )

assert {row["seed"] for row in run_records} == EXPECTED_SEEDS
assert manifest_rows_total == 95
assert sha_rows_total == 95
assert checkpoint_count == 6

audit_path = AUDIT_DIR / (
    "phase3e_cell02_manifest_checkpoint_integrity.json"
)
write_json(
    audit_path,
    {
        "schema_version": "phase3e_cell02_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "run_count": len(run_records),
        "manifest_rows_verified": manifest_rows_total,
        "sha256sums_rows_verified": sha_rows_total,
        "checkpoints_verified": checkpoint_count,
        "quality_gates_passed": 3,
        "run_records": run_records,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03E CELL 2 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher A runs            : 3 / 3 VERIFIED")
print("Manifest rows             : 95 / 95")
print("SHA256SUMS rows           : 95 / 95")
print("Checkpoints               : 6 / 6 VERIFIED")
print("Checkpoint containers     : 6 / 6 VALID")
print("Quality gates             : 3 / 3 PASSED")
print("Selection split           : Validation only")
print("Test Set status           : SEALED")
print("Progress                  : 2 / 7 cells passed")
print("Next                      : audit logs and Validation")

PHASE 03E CELL 2 STATUS
Status                    : PASSED
Teacher A runs            : 3 / 3 VERIFIED
Manifest rows             : 95 / 95
SHA256SUMS rows           : 95 / 95
Checkpoints               : 6 / 6 VERIFIED
Checkpoint containers     : 6 / 6 VALID
Quality gates             : 3 / 3 PASSED
Selection split           : Validation only
Test Set status           : SEALED
Progress                  : 2 / 7 cells passed
Next                      : audit logs and Validation


## Cell 3 — Optimizer log, epoch history và Validation audit

### Công việc

- xác minh 35.420 optimizer row mỗi seed;
- kiểm tra global step liên tục, không NaN/Inf;
- kiểm tra 20 epoch history và 20 Validation artifact mỗi seed;
- đối chiếu best result với epoch history;
- tổng hợp FP16 retry và thời gian train.

### Thời gian dự kiến

30–120 giây.

### Tiến độ hiện tại

Cell 3/7 — kiểm tra log huấn luyện và Validation.

### Đầu ra dự kiến

```text
Status                    : PASSED
Optimizer rows            : 106260
Validation artifacts      : 60
Nonfinite values          : 0
Overflow retries          : 20
Progress                  : 3 / 7 cells passed
```


In [3]:
CELL_STARTED_AT = time.perf_counter()
optimizer_rows_total = 0
validation_artifacts_total = 0
audit_files_total = 0
overflow_retries_total = 0

for run in run_records:
    seed = run["seed"]
    run_root = Path(run["run_root"])
    history_path = (
        run_root
        / "logs"
        / f"teacher_a_seed{seed}_epoch_history.csv"
    )
    with open(
        history_path, "r", encoding="utf-8", newline=""
    ) as source:
        history = list(csv.DictReader(source))
    assert [int(row["epoch"]) for row in history] == list(
        range(1, 21)
    )
    numeric_columns = [
        "training_mean_token_ce",
        "teacher_forced_ce",
        "BLEU-1",
        "BLEU-4",
        "METEOR",
        "ROUGE-L",
        "CIDEr-D",
    ]
    assert all(
        math.isfinite(float(row[column]))
        for row in history
        for column in numeric_columns
    )
    best_history = history[run["best_epoch"] - 1]
    assert math.isclose(
        float(best_history["CIDEr-D"]),
        run["cider_d"],
        rel_tol=0.0,
        abs_tol=1e-12,
    )
    assert math.isclose(
        float(best_history["teacher_forced_ce"]),
        run["validation_ce"],
        rel_tol=0.0,
        abs_tol=1e-12,
    )

    validation_files = sorted(
        (run_root / "validation").glob(
            "epoch_*_validation.json"
        )
    )
    assert len(validation_files) == 20
    for validation_path in validation_files:
        validation = read_json(validation_path)
        assert validation["prediction_count"] == 5000
        assert validation["test_set_status"] == "sealed"
        assert validation["test_set_used"] is False
        assert math.isfinite(
            float(validation["teacher_forced_ce"])
        )
        assert all(
            math.isfinite(float(value))
            for value in validation["metrics"].values()
        )
    validation_artifacts_total += len(validation_files)

    log_path = (
        run_root
        / "logs"
        / f"teacher_a_seed{seed}_optimizer_steps.jsonl"
    )
    log_count = 0
    previous_step = None
    last_log = None
    max_retry_count = 0
    max_retry_per_step = 0
    with open(log_path, "r", encoding="utf-8") as source:
        for line in source:
            if not line.strip():
                continue
            item = json.loads(line)
            log_count += 1
            step = int(item["global_step"])
            if previous_step is not None:
                assert step == previous_step + 1
            previous_step = step
            assert item["test_set_used"] is False
            for key in [
                "mean_token_ce",
                "gradient_norm",
                "scaler_scale_before",
                "scaler_scale_after",
            ]:
                assert math.isfinite(float(item[key]))
            max_retry_count = max(
                max_retry_count,
                int(item.get("overflow_retry_count", 0)),
            )
            max_retry_per_step = max(
                max_retry_per_step,
                int(
                    item.get(
                        "overflow_retries_for_step", 0
                    )
                ),
            )
            last_log = item
    assert log_count == 35420
    assert last_log is not None
    assert int(last_log["global_step"]) == 35420
    assert int(last_log["epoch"]) == 20
    assert max_retry_per_step <= 10
    optimizer_rows_total += log_count
    overflow_retries_total += max_retry_count
    run["overflow_retries"] = max_retry_count
    run["maximum_retry_per_step"] = max_retry_per_step

    full_training_audit = read_json(
        run_root
        / "audits"
        / "phase3d_cell07_full_training.json"
    )
    assert full_training_audit["status"] == "passed"
    assert full_training_audit["test_set_status"] == "sealed"
    assert full_training_audit["test_set_used"] is False
    run["training_seconds"] = float(
        full_training_audit["training_seconds"]
    )

    audit_files = sorted((run_root / "audits").glob("*.json"))
    assert len(audit_files) == 7
    for audit_file in audit_files:
        audit = read_json(audit_file)
        assert audit["status"] == "passed"
        assert audit["test_set_status"] == "sealed"
        assert audit["test_set_used"] is False
    audit_files_total += len(audit_files)

assert optimizer_rows_total == 106260
assert validation_artifacts_total == 60
assert audit_files_total == 21
assert overflow_retries_total == 20

audit_path = AUDIT_DIR / (
    "phase3e_cell03_log_validation_integrity.json"
)
write_json(
    audit_path,
    {
        "schema_version": "phase3e_cell03_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "optimizer_rows_verified": optimizer_rows_total,
        "validation_artifacts_verified": (
            validation_artifacts_total
        ),
        "audit_files_verified": audit_files_total,
        "overflow_retries_total": overflow_retries_total,
        "nonfinite_values": 0,
        "step_gaps": 0,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03E CELL 3 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Optimizer rows            : 106260 / 106260")
print("Validation artifacts      : 60 / 60")
print("Audit files               : 21 / 21 PASSED")
print("Nonfinite values          : 0")
print("Global-step gaps          : 0")
print("Overflow retries          :", overflow_retries_total)
print("Maximum retry per step    :", max(
    row["maximum_retry_per_step"] for row in run_records
))
print("Test Set status           : SEALED")
print("Progress                  : 3 / 7 cells passed")
print("Next                      : aggregate and compare metrics")

PHASE 03E CELL 3 STATUS
Status                    : PASSED
Optimizer rows            : 106260 / 106260
Validation artifacts      : 60 / 60
Audit files               : 21 / 21 PASSED
Nonfinite values          : 0
Global-step gaps          : 0
Overflow retries          : 20
Maximum retry per step    : 1
Test Set status           : SEALED
Progress                  : 3 / 7 cells passed
Next                      : aggregate and compare metrics


## Cell 4 — Three-seed aggregate và Student comparison

### Công việc

- tính mean, sample SD và CV cho sáu chỉ số;
- so sánh Teacher A với S1/S2/S3;
- kiểm tra Teacher A vượt Student mạnh nhất;
- ghi bảng per-run, aggregate và comparison.

### Thời gian dự kiến

1–5 giây.

### Tiến độ hiện tại

Cell 4/7 — tổng hợp ba seed và đối chiếu baseline.

### Đầu ra dự kiến

```text
Teacher A CIDEr-D         : 1.160464 +/- 0.007479
Teacher A vs strongest S3 : +7.97%
Stable across seeds      : True
Progress                  : 4 / 7 cells passed
```


In [4]:
CELL_STARTED_AT = time.perf_counter()
metric_fields = [
    "bleu_1",
    "bleu_4",
    "meteor",
    "rouge_l",
    "cider_d",
    "validation_ce",
]
aggregate = {
    "teacher_id": "A",
    "teacher": (
        "CLIP ViT-B/32 + visual adapter + "
        "shared caption decoder"
    ),
    "seeds": [42, 123, 2026],
    "n": 3,
}
for field in metric_fields:
    values = [
        float(row[field])
        for row in sorted(
            run_records, key=lambda item: item["seed"]
        )
    ]
    mean = statistics.mean(values)
    sample_sd = statistics.stdev(values)
    aggregate[f"{field}_mean"] = mean
    aggregate[f"{field}_sample_sd"] = sample_sd
    aggregate[f"{field}_minimum"] = min(values)
    aggregate[f"{field}_maximum"] = max(values)
    aggregate[f"{field}_cv_percent"] = (
        sample_sd / mean * 100.0
    )

aggregate["training_hours_mean"] = statistics.mean(
    row["training_seconds"] / 3600.0
    for row in run_records
)
aggregate["training_hours_sample_sd"] = statistics.stdev(
    row["training_seconds"] / 3600.0
    for row in run_records
)
aggregate["training_hours_total"] = sum(
    row["training_seconds"] / 3600.0
    for row in run_records
)
aggregate["overflow_retries_total"] = sum(
    row["overflow_retries"] for row in run_records
)
aggregate["best_epochs"] = sorted(
    int(row["best_epoch"]) for row in run_records
)
aggregate["stable_across_seeds"] = (
    aggregate["cider_d_cv_percent"] < 1.0
    and all(row["quality_margin"] > 0 for row in run_records)
)
assert aggregate["stable_across_seeds"] is True
assert aggregate["best_epochs"] == [19, 19, 19]

phase2_per_run_path = exactly_one(
    "phase2_three_seed_per_run.csv"
)
with open(
    phase2_per_run_path,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    student_runs = list(csv.DictReader(source))
assert len(student_runs) == 9

student_by_key = {
    (row["student_id"], int(row["seed"])): row
    for row in student_runs
}
comparisons = []
for teacher_run in sorted(
    run_records, key=lambda row: row["seed"]
):
    for student_id in ["S1", "S2", "S3"]:
        student = student_by_key[
            (student_id, teacher_run["seed"])
        ]
        student_cider = float(student["cider_d"])
        delta = teacher_run["cider_d"] - student_cider
        comparisons.append(
            {
                "seed": teacher_run["seed"],
                "student_id": student_id,
                "student_name": student["student_name"],
                "student_cider_d": student_cider,
                "teacher_a_cider_d": teacher_run["cider_d"],
                "absolute_delta": delta,
                "relative_improvement_percent": (
                    delta / student_cider * 100.0
                ),
            }
        )
        assert delta > 0

student_aggregate_path = exactly_one(
    "phase2_three_seed_aggregate.csv"
)
with open(
    student_aggregate_path,
    "r",
    encoding="utf-8",
    newline="",
) as source:
    student_aggregates = list(csv.DictReader(source))
assert len(student_aggregates) == 3
strongest_student = max(
    student_aggregates,
    key=lambda row: float(row["cider_d_mean"]),
)
strongest_cider = float(strongest_student["cider_d_mean"])
teacher_vs_strongest_delta = (
    aggregate["cider_d_mean"] - strongest_cider
)
teacher_vs_strongest_percent = (
    teacher_vs_strongest_delta / strongest_cider * 100.0
)
assert strongest_student["student_id"] == "S3"
assert teacher_vs_strongest_delta > 0

PER_RUN_CSV = RESULT_DIR / "teacher_a_three_seed_per_run.csv"
per_run_fields = [
    "teacher_id",
    "seed",
    "completed_epochs",
    "optimizer_steps",
    "best_epoch",
    "validation_ce",
    "bleu_1",
    "bleu_4",
    "meteor",
    "rouge_l",
    "cider_d",
    "overflow_retries",
    "maximum_retry_per_step",
    "quality_threshold",
    "quality_margin",
    "test_set_status",
    "test_set_used",
]
with open(
    PER_RUN_CSV, "w", encoding="utf-8", newline=""
) as target:
    writer = csv.DictWriter(
        target, fieldnames=per_run_fields
    )
    writer.writeheader()
    for row in sorted(
        run_records, key=lambda item: item["seed"]
    ):
        writer.writerow(
            {key: row[key] for key in per_run_fields}
        )

AGGREGATE_JSON = RESULT_DIR / (
    "teacher_a_three_seed_aggregate.json"
)
write_json(
    AGGREGATE_JSON,
    {
        "schema_version": "teacher_a_three_seed_aggregate_v1",
        "created_at_utc": now_utc(),
        "status": "verified",
        **aggregate,
        "strongest_student": strongest_student,
        "teacher_vs_strongest_cider_delta": (
            teacher_vs_strongest_delta
        ),
        "teacher_vs_strongest_relative_percent": (
            teacher_vs_strongest_percent
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

COMPARISON_CSV = RESULT_DIR / (
    "teacher_a_vs_students_per_seed.csv"
)
with open(
    COMPARISON_CSV, "w", encoding="utf-8", newline=""
) as target:
    writer = csv.DictWriter(
        target, fieldnames=list(comparisons[0].keys())
    )
    writer.writeheader()
    writer.writerows(comparisons)

audit_path = AUDIT_DIR / (
    "phase3e_cell04_aggregate_comparison.json"
)
write_json(
    audit_path,
    {
        "schema_version": "phase3e_cell04_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "aggregate": aggregate,
        "strongest_student_id": strongest_student[
            "student_id"
        ],
        "teacher_vs_strongest_cider_delta": (
            teacher_vs_strongest_delta
        ),
        "teacher_vs_strongest_relative_percent": (
            teacher_vs_strongest_percent
        ),
        "all_per_seed_student_comparisons_positive": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03E CELL 4 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher A CIDEr-D         :", (
    f"{aggregate['cider_d_mean']:.6f} +/- "
    f"{aggregate['cider_d_sample_sd']:.6f}"
))
print("Teacher A BLEU-4          :", (
    f"{aggregate['bleu_4_mean']:.6f} +/- "
    f"{aggregate['bleu_4_sample_sd']:.6f}"
))
print("Teacher A CE              :", (
    f"{aggregate['validation_ce_mean']:.6f} +/- "
    f"{aggregate['validation_ce_sample_sd']:.6f} nats/token"
))
print("Best epochs               :", aggregate["best_epochs"])
print("Strongest Student         : S3 TinyViT-5M")
print("Teacher vs strongest S3   :", (
    f"+{teacher_vs_strongest_percent:.2f}% CIDEr-D"
))
print("Stable across seeds       :", (
    aggregate["stable_across_seeds"]
))
print("Test Set status           : SEALED")
print("Progress                  : 4 / 7 cells passed")
print("Next                      : build Teacher A model card")

PHASE 03E CELL 4 STATUS
Status                    : PASSED
Teacher A CIDEr-D         : 1.160464 +/- 0.007479
Teacher A BLEU-4          : 0.370974 +/- 0.002636
Teacher A CE              : 2.101002 +/- 0.001440 nats/token
Best epochs               : [19, 19, 19]
Strongest Student         : S3 TinyViT-5M
Teacher vs strongest S3   : +7.97% CIDEr-D
Stable across seeds       : True
Test Set status           : SEALED
Progress                  : 4 / 7 cells passed
Next                      : build Teacher A model card


## Cell 5 — Teacher A model card và KD compatibility

### Công việc

- khóa nguồn pretrained, kiến trúc, dữ liệu và preprocessing;
- ghi mean ± sample SD;
- đăng ký ba best checkpoint;
- khóa logit/feature/sequence KD compatibility;
- ghi rõ giới hạn: chỉ có kết quả Validation.

### Thời gian dự kiến

1–5 giây.

### Tiến độ hiện tại

Cell 5/7 — đóng gói model card và khả năng dùng cho KD.

### Đầu ra dự kiến

```text
Teacher A model card      : WRITTEN
Best checkpoints          : 3
KD channels               : logit / feature / sequence
Qualified for KD          : True
Progress                  : 5 / 7 cells passed
```


In [5]:
CELL_STARTED_AT = time.perf_counter()
MODEL_CARD_PATH = MODEL_CARD_DIR / "teacher_a_model_card.md"
checkpoint_registry = []
for row in sorted(run_records, key=lambda item: item["seed"]):
    checkpoint_registry.append(
        {
            "seed": row["seed"],
            "best_epoch": row["best_epoch"],
            "validation_cider_d": row["cider_d"],
            "validation_bleu_4": row["bleu_4"],
            "validation_ce": row["validation_ce"],
            "checkpoint_path": row["best_checkpoint"]["path"],
            "checkpoint_sha256": row[
                "best_checkpoint"
            ]["sha256"],
            "checkpoint_size_bytes": row[
                "best_checkpoint"
            ]["size_bytes"],
        }
    )

model_card = f"""# Teacher A Model Card

## Identity

- Teacher: CLIP ViT-B/32 + adapter 768→256 + shared caption decoder.
- Dataset: MS COCO 2014 only.
- Split: Karpathy Train + Restval / Validation / sealed Test.
- Seeds: 42, 123, 2026.
- Best checkpoint selection: Validation CIDEr-D.

## Pretrained source

- Identifier: {TEACHER_A_SOURCE_CONTRACT['model_identifier']}.
- Image size: 224.
- Patch size: 32.
- Hidden size: 768.
- Visual tokens after CLS removal: 49.
- Network download during training: false.

## Three-seed Validation

- CIDEr-D: {aggregate['cider_d_mean']:.9f} ± {aggregate['cider_d_sample_sd']:.9f}.
- BLEU-4: {aggregate['bleu_4_mean']:.9f} ± {aggregate['bleu_4_sample_sd']:.9f}.
- METEOR: {aggregate['meteor_mean']:.9f} ± {aggregate['meteor_sample_sd']:.9f}.
- ROUGE-L: {aggregate['rouge_l_mean']:.9f} ± {aggregate['rouge_l_sample_sd']:.9f}.
- CE: {aggregate['validation_ce_mean']:.9f} ± {aggregate['validation_ce_sample_sd']:.9f} nats/token.

## KD compatibility

- Logit KD: allowed because Teacher A uses the locked shared vocabulary and token IDs.
- Feature KD: allowed through an explicit learned projection and feature-normalization contract.
- Sequence KD: allowed using generated captions and sequence scores.

## Limitations

- Metrics are Validation results, not Held-out Test results.
- Checkpoint comparison is valid only under the same preprocessing,
  vocabulary, decoding and metric backend.
- Teacher A is larger than mobile Students and is intended as an offline
  teacher, not the Android deployment model.

## Data leakage boundary

- Held-out Test status: SEALED.
- Test used for selection: false.
- Flickr used: false.
"""
MODEL_CARD_PATH.write_text(model_card, encoding="utf-8")

REGISTRY_PATH = CONFIG_DIR / (
    "teacher_a_three_seed_checkpoint_registry.json"
)
write_json(
    REGISTRY_PATH,
    {
        "schema_version": (
            "teacher_a_three_seed_checkpoint_registry_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "locked",
        "teacher_id": "A",
        "pretrained_source_contract": {
            "model_identifier": TEACHER_A_SOURCE_CONTRACT[
                "model_identifier"
            ],
            "source_files": TEACHER_A_SOURCE_CONTRACT[
                "source_files"
            ],
        },
        "checkpoints": checkpoint_registry,
        "aggregate_metrics": aggregate,
        "kd_channels": {
            "logit": "allowed_shared_vocabulary",
            "feature": "allowed_with_projection_contract",
            "sequence": "allowed",
        },
        "qualified_for_kd": True,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

audit_path = AUDIT_DIR / (
    "phase3e_cell05_teacher_a_model_card_registry.json"
)
write_json(
    audit_path,
    {
        "schema_version": "phase3e_cell05_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "model_card": str(MODEL_CARD_PATH),
        "registry": str(REGISTRY_PATH),
        "best_checkpoint_count": len(checkpoint_registry),
        "kd_channels": ["logit", "feature", "sequence"],
        "qualified_for_kd": True,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03E CELL 5 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher A model card      : WRITTEN")
print("Best checkpoints          :", len(checkpoint_registry))
print("KD channels               : logit / feature / sequence")
print("Qualified for KD          : True")
print("Test Set status           : SEALED")
print("Progress                  : 5 / 7 cells passed")
print("Next                      : lock Teacher B entry contract")

PHASE 03E CELL 5 STATUS
Status                    : PASSED
Teacher A model card      : WRITTEN
Best checkpoints          : 3
KD channels               : logit / feature / sequence
Qualified for KD          : True
Test Set status           : SEALED
Progress                  : 5 / 7 cells passed
Next                      : lock Teacher B entry contract


## Cell 6 — Teacher B BLIP entry contract

### Công việc

- ánh xạ Teacher B với S2 MobileViT-S;
- chọn ứng viên `Salesforce/blip-image-captioning-base`;
- khóa yêu cầu nguồn offline, tokenizer và preprocessing;
- cấm logit KD cho tới khi chứng minh vocabulary alignment;
- cho phép feature KD và sequence KD;
- khóa quality gate theo từng seed.

### Thời gian dự kiến

1–5 giây.

### Tiến độ hiện tại

Cell 6/7 — khóa điều kiện đầu vào cho Teacher B.

### Đầu ra dự kiến

```text
Teacher B candidate       : Salesforce/blip-image-captioning-base
Matched Student           : S2 MobileViT-S
Logit KD                  : PROHIBITED pending alignment
Feature / Sequence KD     : ALLOWED
Progress                  : 6 / 7 cells passed
```


In [6]:
CELL_STARTED_AT = time.perf_counter()
teacher_entry_contract_path = exactly_one(
    "phase3_teacher_pool_entry_contract.json"
)
teacher_entry_contract = read_json(
    teacher_entry_contract_path
)
teacher_b_entry = teacher_entry_contract[
    "teacher_families"
]["B"]
assert teacher_b_entry["matched_student_id"] == "S2"
assert "logit" not in teacher_b_entry["default_kd_channels"]
assert set(teacher_b_entry["default_kd_channels"]) == {
    "feature",
    "sequence",
}

s2_targets = {}
for seed in sorted(EXPECTED_SEEDS):
    student = student_by_key[("S2", seed)]
    cider = float(student["cider_d"])
    s2_targets[str(seed)] = {
        "matched_student_validation_cider_d": cider,
        "minimum_absolute_improvement": 0.001,
        "teacher_minimum_validation_cider_d": cider + 0.001,
    }

TEACHER_B_ENTRY_CONTRACT_PATH = CONFIG_DIR / (
    "teacher_b_blip_entry_contract.json"
)
teacher_b_contract = {
    "contract_version": "phase3_teacher_b_blip_entry_v1",
    "created_at_utc": now_utc(),
    "status": "locked",
    "teacher_id": "B",
    "teacher_name": "BLIP caption teacher",
    "candidate_model_identifier": (
        "Salesforce/blip-image-captioning-base"
    ),
    "candidate_status": (
        "must be verified offline in Phase 03F before use"
    ),
    "matched_student": {
        "student_id": "S2",
        "student_name": "MobileViT-S",
    },
    "dataset": {
        "name": "MS COCO 2014 only",
        "split_protocol": "Karpathy",
        "training_images": 113287,
        "validation_images": 5000,
        "validation_references": 25010,
        "test_set_status": "sealed",
    },
    "required_phase03f_preflight": [
        "lock exact upstream revision and SHA-256",
        "verify config and checkpoint offline",
        "lock BLIP image processor",
        "lock native tokenizer and special token IDs",
        "audit maximum sequence length",
        "run deterministic forward and generation smoke tests",
        "run one-batch overfit and FP16 preflight",
        "prove checkpoint resume equivalence",
    ],
    "kd_compatibility": {
        "logit_kd": (
            "prohibited until BLIP native tokenizer logits are "
            "formally aligned with the shared word vocabulary"
        ),
        "feature_kd": (
            "allowed after locking layer, normalization and "
            "projection contracts"
        ),
        "sequence_kd": (
            "allowed using generated caption text and sequence "
            "scores after deterministic decoding is verified"
        ),
    },
    "validation": {
        "selection_split": "validation",
        "primary_metric": "CIDEr-D",
        "secondary_metrics": [
            "BLEU-1",
            "BLEU-4",
            "METEOR",
            "ROUGE-L",
        ],
        "quality_targets_by_seed": s2_targets,
    },
    "teacher_a_dependency": {
        "three_seed_registry": str(REGISTRY_PATH),
        "teacher_a_qualified_for_kd": True,
    },
    "training_started": False,
    "model_loaded": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "flickr_used": False,
    "authorized_next_action": (
        "Create and run Phase 03F Teacher B BLIP offline source "
        "and architecture preflight. Do not start full training "
        "until all Phase 03F gates pass."
    ),
}
write_json(
    TEACHER_B_ENTRY_CONTRACT_PATH, teacher_b_contract
)

audit_path = AUDIT_DIR / (
    "phase3e_cell06_teacher_b_entry_contract.json"
)
write_json(
    audit_path,
    {
        "schema_version": "phase3e_cell06_v1",
        "created_at_utc": now_utc(),
        "status": "passed",
        "teacher_b_candidate": teacher_b_contract[
            "candidate_model_identifier"
        ],
        "matched_student_id": "S2",
        "quality_targets_by_seed": s2_targets,
        "logit_kd": "prohibited_pending_alignment",
        "feature_kd": "allowed_with_contract",
        "sequence_kd": "allowed",
        "model_loaded": False,
        "training_started": False,
        "test_set_status": "sealed",
        "test_set_used": False,
        "elapsed_seconds": time.perf_counter() - CELL_STARTED_AT,
    },
)

print("=" * 72)
print("PHASE 03E CELL 6 STATUS")
print("=" * 72)
print("Status                    : PASSED")
print("Teacher B candidate       : Salesforce/blip-image-captioning-base")
print("Matched Student           : S2 MobileViT-S")
print("Quality targets           : 3 seeds LOCKED")
print("Logit KD                  : PROHIBITED pending alignment")
print("Feature KD                : ALLOWED with contract")
print("Sequence KD               : ALLOWED")
print("Model loaded              : False")
print("Training started          : False")
print("Test Set status           : SEALED")
print("Progress                  : 6 / 7 cells passed")
print("Next                      : package Phase 03E")

PHASE 03E CELL 6 STATUS
Status                    : PASSED
Teacher B candidate       : Salesforce/blip-image-captioning-base
Matched Student           : S2 MobileViT-S
Quality targets           : 3 seeds LOCKED
Logit KD                  : PROHIBITED pending alignment
Feature KD                : ALLOWED with contract
Sequence KD               : ALLOWED
Model loaded              : False
Training started          : False
Test Set status           : SEALED
Progress                  : 6 / 7 cells passed
Next                      : package Phase 03E


## Cell 7 — Package Phase 03E

### Công việc

- tạo manifest CSV/JSON/SHA256SUMS;
- tạo Teacher A three-seed COMPLETE marker;
- tạo Teacher B preflight AUTHORIZED marker;
- xác minh toàn bộ output;
- không tải BLIP và không mở Test.

### Thời gian dự kiến

5–30 giây.

### Tiến độ hiện tại

Cell 7/7 — đóng gói Phase 03E và phát hành marker.

### Đầu ra bắt buộc

```text
Status                    : COMPLETE
Teacher A three-seed      : VERIFIED
Teacher B preflight       : AUTHORIZED
Manifest failures         : 0
Test Set status           : SEALED
Progress                  : 7 / 7 cells passed
```


In [7]:
CELL_STARTED_AT = time.perf_counter()
MANIFEST_CSV = MANIFEST_DIR / "phase3e_manifest.csv"
MANIFEST_JSON = MANIFEST_DIR / "phase3e_manifest.json"
SHA256SUMS = MANIFEST_DIR / "phase3e_SHA256SUMS.txt"
TEACHER_A_COMPLETE_MARKER = OUTPUT_ROOT / (
    "PHASE3_TEACHER_A_THREE_SEED_COMPLETE.json"
)
TEACHER_B_AUTHORIZED_MARKER = OUTPUT_ROOT / (
    "PHASE3_TEACHER_B_PREFLIGHT_AUTHORIZED.json"
)
excluded = {
    MANIFEST_CSV.resolve(),
    MANIFEST_JSON.resolve(),
    SHA256SUMS.resolve(),
    TEACHER_A_COMPLETE_MARKER.resolve(),
    TEACHER_B_AUTHORIZED_MARKER.resolve(),
}
rows = []
for artifact in sorted(
    OUTPUT_ROOT.rglob("*"),
    key=lambda path: path.as_posix(),
):
    if not artifact.is_file():
        continue
    if artifact.resolve() in excluded:
        continue
    assert not artifact.name.endswith(".tmp")
    rows.append(
        {
            "relative_path": artifact.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "size_bytes": artifact.stat().st_size,
            "sha256": sha256_file(artifact),
        }
    )

with open(
    MANIFEST_CSV, "w", encoding="utf-8", newline=""
) as target:
    writer = csv.DictWriter(
        target,
        fieldnames=["relative_path", "size_bytes", "sha256"],
    )
    writer.writeheader()
    writer.writerows(rows)
write_json(
    MANIFEST_JSON,
    {
        "schema_version": "phase3e_manifest_v1",
        "created_at_utc": now_utc(),
        "artifact_count": len(rows),
        "total_size_bytes": sum(
            row["size_bytes"] for row in rows
        ),
        "artifacts": rows,
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
SHA256SUMS.write_text(
    "".join(
        f"{row['sha256']}  {row['relative_path']}\n"
        for row in rows
    ),
    encoding="utf-8",
)

with open(
    MANIFEST_CSV, "r", encoding="utf-8", newline=""
) as source:
    verified = list(csv.DictReader(source))
for row in verified:
    artifact = resolve_inside(
        OUTPUT_ROOT, row["relative_path"]
    )
    assert artifact.stat().st_size == int(row["size_bytes"])
    assert sha256_file(artifact) == row["sha256"]

common_manifests = {
    "csv": {
        "relative_path": MANIFEST_CSV.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(MANIFEST_CSV),
    },
    "json": {
        "relative_path": MANIFEST_JSON.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(MANIFEST_JSON),
    },
    "sha256sums": {
        "relative_path": SHA256SUMS.relative_to(
            OUTPUT_ROOT
        ).as_posix(),
        "sha256": sha256_file(SHA256SUMS),
    },
}
write_json(
    TEACHER_A_COMPLETE_MARKER,
    {
        "marker_version": (
            "phase3_teacher_a_three_seed_complete_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "complete",
        "teacher_id": "A",
        "seeds": [42, 123, 2026],
        "run_count": 3,
        "completed_epochs_per_run": 20,
        "optimizer_steps_per_run": 35420,
        "aggregate_metrics": aggregate,
        "checkpoint_registry": {
            "relative_path": REGISTRY_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(REGISTRY_PATH),
        },
        "model_card": {
            "relative_path": MODEL_CARD_PATH.relative_to(
                OUTPUT_ROOT
            ).as_posix(),
            "sha256": sha256_file(MODEL_CARD_PATH),
        },
        "quality_gate_passed_seeds": 3,
        "stable_across_seeds": True,
        "qualified_for_kd": True,
        "manifests": common_manifests,
        "selection_split": "validation",
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)
write_json(
    TEACHER_B_AUTHORIZED_MARKER,
    {
        "marker_version": (
            "phase3_teacher_b_preflight_authorized_v1"
        ),
        "created_at_utc": now_utc(),
        "status": "ready",
        "teacher_id": "B",
        "candidate_model_identifier": (
            "Salesforce/blip-image-captioning-base"
        ),
        "entry_contract": {
            "relative_path": (
                TEACHER_B_ENTRY_CONTRACT_PATH.relative_to(
                    OUTPUT_ROOT
                ).as_posix()
            ),
            "sha256": sha256_file(
                TEACHER_B_ENTRY_CONTRACT_PATH
            ),
        },
        "teacher_a_three_seed_complete": {
            "relative_path": (
                TEACHER_A_COMPLETE_MARKER.relative_to(
                    OUTPUT_ROOT
                ).as_posix()
            ),
            "sha256": sha256_file(
                TEACHER_A_COMPLETE_MARKER
            ),
        },
        "manifests": common_manifests,
        "model_loaded": False,
        "training_started": False,
        "authorized_next_action": (
            "Create Phase 03F Teacher B BLIP offline source, "
            "tokenizer, preprocessing and architecture preflight."
        ),
        "test_set_status": "sealed",
        "test_set_used": False,
    },
)

print("=" * 72)
print("PHASE 03E CELL 7 STATUS")
print("=" * 72)
print("Status                    : COMPLETE")
print("Teacher A three-seed      : VERIFIED")
print("Teacher A CIDEr-D         :", (
    f"{aggregate['cider_d_mean']:.6f} +/- "
    f"{aggregate['cider_d_sample_sd']:.6f}"
))
print("Teacher A qualified KD    : True")
print("Teacher B preflight       : AUTHORIZED")
print("Teacher B model loaded    : False")
print("Manifest rows             :", len(rows))
print("Manifest failures         : 0")
print("Test Set status           : SEALED")
print("Test Set used             : False")
print("Progress                  : 7 / 7 cells passed")
print("Next                      : Phase 03F Teacher B BLIP preflight")

PHASE 03E CELL 7 STATUS
Status                    : COMPLETE
Teacher A three-seed      : VERIFIED
Teacher A CIDEr-D         : 1.160464 +/- 0.007479
Teacher A qualified KD    : True
Teacher B preflight       : AUTHORIZED
Teacher B model loaded    : False
Manifest rows             : 12
Manifest failures         : 0
Test Set status           : SEALED
Test Set used             : False
Progress                  : 7 / 7 cells passed
Next                      : Phase 03F Teacher B BLIP preflight
